In [ ]:
# Importer les bibliothèques necessaires
!pip install gdown
import gdown
import pandas as pd
import re
import gc
import os
import sys
import pandas as pd
import numpy as np
import tqdm
import seaborn as sns

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from scipy.stats import chi2_contingency
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, roc_curve, auc, 
classification_report, confusion_matrix, roc_auc_score

In [ ]:
np.random.seed(123)

In [ ]:
# Charger les données
df = pd.read_csv('Fraud Detection Dataset.csv')  # changer le nom du fichier si nécessaire
print('Forme du dataset chargé :', df.shape)
print(df['Fraudulent'].value_counts())


1ère méthode de remplissage des Na : moyenne

Retourne le dataset "df_mean"

In [ ]:
# Séparation des colonnes numériques et catégorielles
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop('Fraudulent')
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

# Création d'une copie de sécurité
df_mean = df.copy()

# Remplacement des valeurs manquantes pour les variables numériques
for col in numeric_cols:
    for fraud_value in [0, 1]:
        mean_value = df_mean.loc[df_mean['Fraudulent'] == fraud_value, col].mean()
        df_mean.loc[
            (df_mean['Fraudulent'] == fraud_value) & (df_mean[col].isna()),
            col
        ] = mean_value

# Remplacement des valeurs manquantes pour les variables catégorielles
for col in categorical_cols:
    for fraud_value in [0, 1]:
        mode_series = df_mean.loc[df_mean['Fraudulent'] == fraud_value, col].mode()
        if not mode_series.empty:
            mode_value = mode_series.iloc[0]
            df_mean.loc[
                (df_mean['Fraudulent'] == fraud_value) & (df_mean[col].isna()),
                col
            ] = mode_value

# Vérification du remplissage
print("Pourcentage de valeurs manquantes après traitement :")
print((df_mean.isnull().mean() * 100).round(3))

2ème méthode de remplissage des Na : interpolation Bayesienne

Retourne le dataset "df_interpolation"

In [ ]:
### Remplir

"""
Le code à rentrer ici prend en entrée "df" déjà définie juste avant et renvoie "df_interpolation"

"""



###____________________


#Remplir code ici 


###____________________




Preprocessing : 
Séparation des variables explicatives X de la variable expliquée y 

In [ ]:
y = df['Fraudulent']
X = df.drop(columns=['Fraudulent'])

I - Sans SMOTE 

Preprocessing :
• Séparation du dataset en train et test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
random_state=42, stratify=y)

Entraînement des classificateurs de chaque modèle + calcul seuil optimal + calcul des métriques + courbe ROC

RandomForest

In [ ]:
# Entraînement du classificateur
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_res, y_res)
y_pred = clf.predict(X_test)
y_score = clf.predict_proba(X_test)[:,1]

# Métriques au seuil de 50%
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

# Métriques au seuil de classification de façon optimale selon l'indice de Youden







print('Précision :', prec)
print('Rappel :', rec)
print('AUC :', roc_auc)
print(classification_report(y_test, y_pred))
print('Matrice de confusion :\n', confusion_matrix(y_test, y_pred))


print('Précision avec Youden :', prec)
print('Rappel avec Youden:', rec)
print(classification_report(y_test, y_pred))


# Tracé de la courbe ROC
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel('Taux de Faux Positifs')
plt.ylabel('Taux de Vrais Positifs')
plt.title('Courbe ROC (AUC = %.3f)' % roc_auc)
plt.show()

Régression Logistique

In [ ]:
# Entraînement du classificateur
lr = LogisticRegression()
lr.fit(X_test,y_test)

#construction de y_pred
y_pred=lr.predict(X_test)

# Métriques au seuil de 50%
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

#Matrice de confusion au seuil de 50%
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred,labels = [1,0])

#Regardons la distribution de probabilité qu'une transaction soit frauduleuse 
y_pred_proba = lr.predict_proba(X_test)

#On peut tracer un histogramme de ces probas
pd.Series(y_pred_proba[:,1]).hist()


"""
fpr : FP rate
tpr : TP rate
ths : threshold

"""

#La fonction roc_curve permet de tracer la courbe ROC 
# à partir d'un vecteur de probabilités de la classe positive (ici (fraudulent=1))
fpr, tpr, ths = roc_curve(t, y_pred_proba[:,1])
auc_score = auc(fpr,tpr)
plt.plot(fpr,tpr,label="AUC Score:" + str(auc_score))
plt.xlabel('FALSE POSITIVE rate',fontsize='15')
plt.ylabel('TRUE POSITIVE rate',fontsize='15')
plt.legend(loc='best')

#Recherche d'un seuil (threshold) idéal 
"""
On souhaiterait maximiser le taux de détection d'une transaction frauduleuse
tout en minimisant le taux de transaction non frauduleuses classées comme tel

On va pour cela agir sur le seuil de classification (threshold)

Dans un premier temps, on cherche un seuil qui maximise le recall (sensitivity 
i.e le TP rate) tout en minimisant le FP rate. Néanmoins, le seuil obtenu n'est pas 
le plus optimal selon les situations. 

On va donc dans un second temps utiliser l'indice de Youden qui va nous donner un 
autre seuil de façon optimale selon l'indice de Youden. 
"""

#Recherche seuil sans indice de Youden

index_max= np.where(tpr==np.max(tpr))[0] #indices pour lesquelle tpr est max
threshold_index=np.max(index_max)
threshold= ths[threshold_index]

#On peut le voir graphiquement
plt.plot(ths,tpr)
plt.xlabel('threshold')
plt.ylabel('TRUE POSITIVE rate')

#On obtient donc de nouvelles valeurs prédites ajustées
from sklearn.preprocessing import binarize
y_pred_th= binarize(y_pred_proba, threshold=threshold)
confusion_mat=confusion_matrix(t, y_pred_th[:,1],labels=[1,0])
    
#Les métriques associées
recall=tpr[threshold_index]
precision=(confusion_mat[0,0])/(confusion_mat[0,0] + confusion_mat[1,0])
specificity=1-fpr[threshold_index]
f1_score=2*precision*recall/(precision+recall)

#Recherche d'un seuil avec l'indice de Youden
"""
On va cette fois-ci maximiser la quantité : recall + specificty - 1 
On en tirera un seuil optimal noté threshold_y
On s'intéressera aux métrques associées à ce seuil afin de les comparer à
celles sans l'indice de Youden

"""
youden= tpr - fpr
index_max_y= np.where(youden==np.max(youden))
threshold_y_index= np.max(index_max_y)
threshold_y=ths[threshold_y_index]


#On obtient de même de nouvelles valeurs prédites ajustées
y_pred_th_y= binarize(y_pred_proba, threshold=threshold_y)
confusion_mat_y = confusion_matrix(t, y_pred_th_y[:,1],labels=[1,0])

#Les métriques associées avec Youden
recall_y=tpr[threshold_y_index]
precision_y=(confusion_mat_y[0,0])/(confusion_mat_y[0,0] + confusion_mat_y[1,0])
specificity_y=1-fpr[threshold_y_index]
f1_score_y=2*precision_y*recall_y/(precision_y+recall_y)

#Affichage des métriques utiles
print("accuracy:" +str(accuracy))
print("AUC: "+ str(auc_score))
print("threshold: " + str(threshold) )
print("recall (sensitivity) : " + str(recall) )
print("precision :" + str(precision))
print("specificity : " + str(specificity) )
print("f1-score avec Youden: " + str(f1_score))
print("threshold avec Youden: " + str(threshold_y) )
print("recall (sensitivity) avec Youden : " + str(recall_y) )
print("precision avec Youden:" + str(precision_y))
print("specificity avec Youden: " + str(specificity_y) )
print("f1-score avec Youden : " + str(f1_score_y))
print("Indice de Youden: " + str(youden[threshold_y_index]))

# Tracé de la courbe ROC
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.xlabel('Taux de Faux Positifs')
plt.ylabel('Taux de Vrais Positifs')
plt.title('Courbe ROC (AUC = %.3f)' % roc_auc)
plt.show()




SVM

LightGBM

In [ ]:
#